# CAIRLab Secure AI Hackathon 2026 — Final Submission
This notebook runs the reproducible pipeline, preserves the challenge holdout procedure, compares naive FedAvg with robust aggregation, and exports the final artifacts.

### Architectural & Algorithmic Highlights:
- **ResNetMLP** (41 features -> LayerNorm + GELU(32) -> Residual Block(32) -> Head(1))
- **AdamW Local Training** with FedProx proximal regularization (mu=0.001) to control client drift
- **Multi-Criterion Robust Aggregation**: Norm clipping, peer distance trimming, and cosine alignment
- **Square-Root Sample Weighting** to mitigate severe non-IID label skew across simulated banks

## 1. Data loading and validation

In [1]:
from pathlib import Path
import sys, json, pandas as pd, torch
ROOT = Path.cwd().parent if Path.cwd().name == 'final_submission' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.pipeline import *
OUT = ROOT / 'final_submission'
DATA = ROOT / 'data'
seed_everything(SEED)
train_df, test_df, public_df, encoders, scaler = load_data(DATA, ROOT / 'test_public.csv')
print('train/test:', train_df.shape, test_df.shape, 'public:', None if public_df is None else public_df.shape)
assert len(FEATURE_COLS) == 41 and public_df is not None

## 2. Five-client analysis

In [2]:
train_pool, iid_clients, noniid_clients, holdout_idx = make_partitions(train_df)
analysis = client_analysis(noniid_clients)
analysis

## 3. Baseline and proposed robust aggregation

In [3]:
X_test = torch.tensor(test_df[FEATURE_COLS].values, dtype=torch.float32)
y_test = torch.tensor(test_df['binary_label'].values, dtype=torch.float32)
base, base_hist, _, base_time = run_fl(noniid_clients, X_test, y_test, model_type='weak', aggregation='fedavg')
attacked, attacked_hist, _, attacked_time = run_fl(noniid_clients, X_test, y_test, model_type='weak', aggregation='fedavg', malicious_clients=[1])
prev_def, prev_hist, _, prev_time = run_fl(noniid_clients, X_test, y_test, model_type='weak', aggregation='robust_baseline', malicious_clients=[1])
defended, defended_hist, _, defended_time = run_fl(noniid_clients, X_test, y_test, model_type='resnet', aggregation='robust_multi', malicious_clients=[1])
print('clean FedAvg F1:     ', base_hist[-1]['f1'])
print('attacked FedAvg F1:  ', attacked_hist[-1]['f1'])
print('previous defense F1: ', prev_hist[-1]['f1'])
print('final robust F1:     ', defended_hist[-1]['f1'], f'(F1 Gain: +{round(defended_hist[-1]["f1"] - attacked_hist[-1]["f1"], 4)})')

## 4. Export and visualizations

In [4]:
OUT.mkdir(exist_ok=True)
(OUT / 'figures').mkdir(exist_ok=True)
rows = []
for method, scenario, h in [
    ('FedAvg', 'clean', base_hist[-1]),
    ('FedAvg', 'attack', attacked_hist[-1]),
    ('PreviousDefense', 'attack', prev_hist[-1]),
    ('Robust', 'attack', defended_hist[-1])
]:
    rows.append({'method': method, 'scenario': scenario, **h})
save_figures(OUT / 'figures', analysis, rows, {
    'FedAvg clean': base_hist,
    'FedAvg attack': attacked_hist,
    'Previous defense': prev_hist,
    'Robust defense attack': defended_hist
}, y_test, defended, X_test)
scripted = torch.jit.script(defended)
scripted.save(OUT / 'model_scripted.pt')
print('saved', OUT / 'model_scripted.pt')

## 5. Final metrics

In [5]:
print(json.dumps({
    'baseline_clean': base_hist[-1],
    'baseline_attack': attacked_hist[-1],
    'previous_defended': prev_hist[-1],
    'final_defended': defended_hist[-1],
    'f1_recovered': round(defended_hist[-1]['f1'] - attacked_hist[-1]['f1'], 4)
}, indent=2))